# SmartScan - train the occupancy predictor on Colab

**SIH 26055 - Smart Scan Strategy for Electronic Warfare**

Trains `predictor_easy` and `predictor_hard` on a Colab GPU, then hands back
checkpoints to drop into `runs/checkpoints/`.

### Why only the predictors

Colab is the right machine for **supervised** training here and the wrong one
for the RL agents:

| | bottleneck | Colab | local (24 cores) |
|---|---|---|---|
| predictor | transformer fwd/bwd on GPU | **much faster** | 0.36 s/batch on CPU |
| dqn / ppo / hybrid | Python env rollouts, 8 parallel | 2 vCPU - **slower** | 24 cores |

The RL agents spend nearly all their time stepping the simulator, not in
matmuls, so a GPU does not help and Colab's 2 vCPUs make it worse. Train those
locally.

### Runtime
**Runtime -> Change runtime type -> T4 GPU**, then Run all.

In [ ]:
import subprocess, sys, os, json, shutil
from pathlib import Path

print('installing smartscan...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'git+https://github.com/shirish-raj-gupta/SIH26055_Prototype.git'],
               check=True)
import smartscan
print('smartscan', smartscan.__version__)

In [ ]:
import torch

# Trust a real kernel launch, not is_available(): a GPU can be present and
# unusable if its compute capability predates the installed torch build.
GPU = False
if torch.cuda.is_available():
    major, minor = torch.cuda.get_device_capability(0)
    print(f'GPU: {torch.cuda.get_device_name(0)}  sm_{major}{minor}')
    try:
        (torch.zeros(8, 8, device='cuda') @ torch.zeros(8, 8, device='cuda')).cpu()
        GPU = True
    except Exception as exc:
        print('GPU present but UNUSABLE:', type(exc).__name__)
else:
    print('no GPU allocated - use Runtime -> Change runtime type -> T4 GPU')
print('torch', torch.__version__, '| device:', 'cuda' if GPU else 'cpu')

## How many episodes fit

`build_windows` materialises one dense float32 array with no streaming path, at
roughly 105 MB per episode. That is what caps the corpus, and it is a RAM limit
rather than a GPU one - so check it rather than guessing. The library refuses
outright if the request will not fit, which is why the local runs used 40.

In [ ]:
from smartscan.agents.predictors import _available_memory_bytes
from smartscan.config import load_config

avail = _available_memory_bytes()
cfg = load_config('medium.yaml')
per_ep = 400 * 4 * cfg.n_channels * cfg.predictor.window_slots * 4
safe = int(0.6 * avail / per_ep) if avail else 40
print(f'available RAM   : {avail/1e9:.1f} GB')
print(f'per episode     : {per_ep/1e6:.0f} MB')
print(f'largest safe run: ~{safe} episodes   (local box managed 40)')

EPISODES = max(16, min(safe, 96))
print(f'using           : {EPISODES} episodes')

## Train

Both tiers, transformer architecture. The epoch shipped is the one with the
best **average precision** on observed labels, not the lowest loss: occupancy
is ~4-9 % positive, so the loss is minimised by predicting "idle" everywhere.
Each checkpoint records its own architecture, so it loads regardless of what
the config happens to default to.

In [ ]:
OUT = Path('/content/checkpoints'); OUT.mkdir(parents=True, exist_ok=True)

for tier in ('easy', 'hard'):
    print(f'{chr(61)*30}  {tier}  {chr(61)*30}', flush=True)
    r = subprocess.run(
        [sys.executable, '-m', 'smartscan.cli', 'train', '--what', 'predictor',
         '--config', f'configs/{tier}.yaml', '--arch', 'transformer',
         '--episodes', str(EPISODES)],
        capture_output=True, text=True)
    print(r.stdout[-3000:])
    if r.returncode != 0:
        print('FAILED:', r.stderr[-2000:])
        continue
    for name in (f'predictor_{tier}.pt', f'predictor_{tier}_history.json'):
        src = Path('runs/checkpoints') / name
        if src.is_file():
            shutil.copy(src, OUT / name)
            print('saved', name, f'{src.stat().st_size/1e6:.1f} MB')

## Did they actually learn?

The scheduler takes an **argmax** over these probabilities - it ranks channels
and never thresholds - so AUC and AP lift over the base rate are what matter.
Accuracy is not: on a 4 % positive rate, predicting "idle" everywhere scores
96 %. A model that never crosses 0.5 can still be perfectly usable.

In [ ]:
for tier in ('easy', 'hard'):
    f = OUT / f'predictor_{tier}_history.json'
    if not f.is_file():
        print(f'{tier}: no history'); continue
    h = json.loads(f.read_text())
    s = h['scores_vs_truth']
    lift = s['average_precision'] / s['positive_rate']
    verdict = 'RANKS' if (s['auc'] >= 0.55 and lift >= 1.2) else 'NO SKILL'
    print(f"{tier:6} AUC {s['auc']:.3f}  AP {s['average_precision']:.3f}  "
          f"base {s['positive_rate']:.3f}  lift {lift:.2f}x  -> {verdict}")
    if 'teacher_scores_vs_truth' in h:
        t = h['teacher_scores_vs_truth']
        print(f"       teacher (privileged upper bound): AUC {t['auc']:.3f} "
              f"AP {t['average_precision']:.3f}")

## Download

Put these in `runs/checkpoints/` in the repo. `hybrid_easy` and `hybrid_hard`
can then be trained locally - hybrid training refuses to start without a
trained predictor for its tier, rather than augmenting with an untrained one
and teaching the policy to read noise.

In [ ]:
from google.colab import files
for f in sorted(OUT.iterdir()):
    print('downloading', f.name)
    files.download(str(f))